# Current RF-repair result audit

## tl;dr

The submission-safe primary result is the value-fault closed loop plus the
component-blind structural strict run: **46/48 simulator-verified recovery,
45/48 component localization, and 37/48 full diagnosis**. The older 42/48
diagnosis total included a structural run whose `RFAULT<component>` names
exposed the answer and must not be used as blind diagnostic evidence.

The blind development ablation and the disjoint within-circuit holdout both
reconcile with their stored summaries. On holdout, the combined interface
improves final diagnosis from 9/18 to 18/18 and recovery from 14/18 to 17/18.
The remaining failure is a validly parsed but physically wrong 1000x L2 value,
which the deterministic SPICE validator rejects.

## Context & Methods

This notebook is the executable audit companion for the two-page ACK draft.
It reads frozen protocols, manifests, trajectory JSON, and saved ngspice
S-parameter tables. It recomputes denominators, diagnosis and recovery counts,
manifest hashes, saved-vs-recomputed simulator verdicts, response hashes where
available, and blind placeholder naming.

### Key Assumptions

- The repository root is the notebook execution directory.
- A trajectory is uniquely identified by `(experiment_id, fault_id, trajectory)`.
- Final localization and diagnosis follow the last attempted iteration, matching
  the experiment runner.
- The primary 48-trajectory headline combines the value run and blind structural
  strict v2; interface-development and holdout runs remain separate.
- This notebook re-evaluates saved S-parameter evidence but does not rerun ngspice.

## Data

### 1. Load frozen protocols and validation code

In [1]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from rf_repair.simulator import _measure, _read_table
from rf_repair.validator import validate_bpf_metrics

SPEC = json.loads((ROOT / "specs/bpf_100mhz.json").read_text(encoding="utf-8"))
PROTOCOL_PATHS = {
    "value_primary": ROOT / "experiments/closed_loop_value_repeat_v1_0.json",
    "blind_strict": ROOT / "experiments/closed_loop_structural_blind_v2.json",
    "blind_unit": ROOT / "experiments/closed_loop_structural_blind_unit_adapter_v2_1.json",
    "blind_label": ROOT / "experiments/closed_loop_structural_blind_label_guidance_v2_2.json",
    "blind_combined": ROOT / "experiments/closed_loop_structural_blind_combined_v2_3.json",
    "holdout_strict": ROOT / "experiments/closed_loop_structural_blind_holdout_strict_v3_0.json",
    "holdout_combined": ROOT / "experiments/closed_loop_structural_blind_holdout_combined_v3_0.json",
}

def table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    lines.extend("| " + " | ".join(str(value) for value in row) + " |" for row in rows)
    return "\n".join(lines)

print("repository:", ROOT)
print("protocol count:", len(PROTOCOL_PATHS))

repository: /Users/taehoon/Desktop/졸업논문2
protocol count: 7


### 2. Audit each experiment from saved evidence

In [2]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def audit_protocol(label: str, protocol_path: Path) -> dict:
    protocol = load_json(protocol_path)
    manifest_path = ROOT / protocol["fault_manifest"]
    manifest_bytes = manifest_path.read_bytes()
    manifest = json.loads(manifest_bytes)
    experiment_root = ROOT / "build/experiments" / protocol["experiment_id"]

    expected_keys = {
        (case["fault_id"], trajectory)
        for case in manifest["cases"]
        for trajectory in range(1, int(protocol["trajectories_per_case"]) + 1)
    }
    observed_keys = []
    trajectories = []
    iterations = []
    missing_files = []
    initial_fault_passes = []
    verdict_mismatches = []
    final_mismatches = []

    for case in manifest["cases"]:
        fault_id = case["fault_id"]
        case_root = experiment_root / fault_id
        faulty_table = case_root / "faulty_simulation/sparams.tsv"
        if not faulty_table.exists():
            missing_files.append(str(faulty_table.relative_to(ROOT)))
        else:
            verdict = validate_bpf_metrics(_measure(_read_table(faulty_table), SPEC), SPEC)
            if verdict["verdict"] == "PASS":
                initial_fault_passes.append(fault_id)

        for trajectory in range(1, int(protocol["trajectories_per_case"]) + 1):
            key = (fault_id, trajectory)
            trajectory_root = case_root / f"trajectory_{trajectory}"
            final_path = trajectory_root / "final.json"
            if not final_path.exists():
                missing_files.append(str(final_path.relative_to(ROOT)))
                continue
            observed_keys.append(key)
            final = load_json(final_path)
            iteration_paths = sorted(
                trajectory_root.glob("iteration_*.json"),
                key=lambda path: int(path.stem.split("_")[1]),
            )
            rows = [load_json(path) for path in iteration_paths]
            if len(rows) != final["iterations_used"]:
                final_mismatches.append((key, "iterations_used"))
            if any(row.get("specification_recovered") is True for row in rows) != final["final_recovered"]:
                final_mismatches.append((key, "final_recovered"))
            if rows[-1].get("component_correct", False) != final["final_component_correct"]:
                final_mismatches.append((key, "final_component_correct"))
            final_diagnosis = final.get(
                "final_diagnosis_correct",
                rows[-1].get("diagnosis_correct", False),
            )
            if rows[-1].get("diagnosis_correct", False) != final_diagnosis:
                final_mismatches.append((key, "final_diagnosis_correct"))

            for row in rows:
                row = dict(row)
                row["experiment_id"] = protocol["experiment_id"]
                iterations.append(row)
                if row.get("patch_accepted") is True:
                    table_path = trajectory_root / f"simulation_{row['iteration']}/sparams.tsv"
                    if not table_path.exists():
                        missing_files.append(str(table_path.relative_to(ROOT)))
                        continue
                    verdict = validate_bpf_metrics(_measure(_read_table(table_path), SPEC), SPEC)
                    recovered = verdict["verdict"] == "PASS"
                    if recovered != row.get("specification_recovered"):
                        verdict_mismatches.append((key, row["iteration"]))

            trajectories.append({
                "fault_id": fault_id,
                "trajectory": trajectory,
                "recovered": bool(final["final_recovered"]),
                "localized": bool(final["final_component_correct"]),
                "diagnosed": bool(final_diagnosis),
                "iterations": int(final["iterations_used"]),
            })

    summary = load_json(experiment_root / "summary.json")
    recomputed = {
        "trajectory_count": len(trajectories),
        "final_recovery_count": sum(row["recovered"] for row in trajectories),
        "final_component_localization_count": sum(row["localized"] for row in trajectories),
        "final_full_diagnosis_count": sum(row["diagnosed"] for row in trajectories),
        "total_agent_calls": len(iterations),
        "accepted_candidate_simulation_calls": sum(
            row.get("patch_accepted") is True for row in iterations
        ),
        "api_or_schema_failure_count": sum(
            row.get("outcome") == "API_OR_SCHEMA_FAILURE" for row in iterations
        ),
    }
    summary_mismatches = {
        key: (summary.get(key), value)
        for key, value in recomputed.items()
        if summary.get(key) != value
    }

    response_ids = [row.get("response_id") for row in iterations if row.get("response_id")]
    hash_rows = [
        row for row in iterations
        if row.get("output_text") is not None and row.get("output_text_sha256") is not None
    ]
    hash_mismatches = [
        (row["fault_id"], row["trajectory"], row["iteration"])
        for row in hash_rows
        if hashlib.sha256(row["output_text"].encode()).hexdigest()
        != row["output_text_sha256"]
    ]

    return {
        "label": label,
        "protocol": protocol,
        "manifest": manifest,
        "experiment_root": experiment_root,
        "expected_keys": expected_keys,
        "observed_keys": observed_keys,
        "trajectories": trajectories,
        "iterations": iterations,
        "manifest_hash_matches": hashlib.sha256(manifest_bytes).hexdigest()
        == protocol["fault_manifest_sha256"],
        "initial_fault_passes": initial_fault_passes,
        "missing_files": missing_files,
        "verdict_mismatches": verdict_mismatches,
        "final_mismatches": final_mismatches,
        "summary_mismatches": summary_mismatches,
        "response_ids": response_ids,
        "response_hash_count": len(hash_rows),
        "hash_mismatches": hash_mismatches,
    }

audits = {
    label: audit_protocol(label, path)
    for label, path in PROTOCOL_PATHS.items()
}
print("audited experiments:", len(audits))
print("audited trajectories:", sum(len(a["trajectories"]) for a in audits.values()))

audited experiments: 7
audited trajectories: 138


## Results

### 3. Submission-safe primary result

In [3]:
primary = audits["value_primary"]["trajectories"] + audits["blind_strict"]["trajectories"]
primary_counts = {
    "trajectories": len(primary),
    "recovery": sum(row["recovered"] for row in primary),
    "localization": sum(row["localized"] for row in primary),
    "full_diagnosis": sum(row["diagnosed"] for row in primary),
}
display(Markdown(table(
    ["Primary component", "Trajectories", "Recovery", "Localization", "Full diagnosis"],
    [
        ["Value closed loop", 30, "30/30", "27/30", "27/30"],
        ["Blind structural strict", 18, "16/18", "18/18", "10/18"],
        ["Combined primary", 48, "46/48 (95.8%)", "45/48 (93.8%)", "37/48 (77.1%)"],
    ],
)))
print(primary_counts)
assert primary_counts == {
    "trajectories": 48,
    "recovery": 46,
    "localization": 45,
    "full_diagnosis": 37,
}

| Primary component | Trajectories | Recovery | Localization | Full diagnosis |
| --- | --- | --- | --- | --- |
| Value closed loop | 30 | 30/30 | 27/30 | 27/30 |
| Blind structural strict | 18 | 16/18 | 18/18 | 10/18 |
| Combined primary | 48 | 46/48 (95.8%) | 45/48 (93.8%) | 37/48 (77.1%) |

{'trajectories': 48, 'recovery': 46, 'localization': 45, 'full_diagnosis': 37}


### 4. Blind interface development and disjoint holdout

In [4]:
comparison_labels = [
    "blind_strict", "blind_unit", "blind_label", "blind_combined",
    "holdout_strict", "holdout_combined",
]
comparison_rows = []
for label in comparison_labels:
    rows = audits[label]["trajectories"]
    comparison_rows.append([
        label,
        len(rows),
        f"{sum(row['recovered'] for row in rows)}/{len(rows)}",
        f"{sum(row['localized'] for row in rows)}/{len(rows)}",
        f"{sum(row['diagnosed'] for row in rows)}/{len(rows)}",
        len(audits[label]["iterations"]),
    ])
display(Markdown(table(
    ["Condition", "Trajectories", "Recovery", "Localization", "Full diagnosis", "Agent calls"],
    comparison_rows,
)))

| Condition | Trajectories | Recovery | Localization | Full diagnosis | Agent calls |
| --- | --- | --- | --- | --- | --- |
| blind_strict | 18 | 16/18 | 18/18 | 10/18 | 22 |
| blind_unit | 18 | 17/18 | 18/18 | 11/18 | 21 |
| blind_label | 18 | 16/18 | 18/18 | 18/18 | 26 |
| blind_combined | 18 | 17/18 | 18/18 | 18/18 | 20 |
| holdout_strict | 18 | 14/18 | 18/18 | 9/18 | 27 |
| holdout_combined | 18 | 17/18 | 18/18 | 18/18 | 20 |

### 5. Completeness, uniqueness, hashes, and simulator reconciliation

In [5]:
integrity_rows = []
all_response_ids = []
for label, audit in audits.items():
    duplicates = len(audit["observed_keys"]) - len(set(audit["observed_keys"]))
    missing_keys = len(audit["expected_keys"] - set(audit["observed_keys"]))
    all_response_ids.extend(audit["response_ids"])
    integrity_rows.append([
        label,
        audit["manifest_hash_matches"],
        duplicates,
        missing_keys,
        len(audit["missing_files"]),
        len(audit["initial_fault_passes"]),
        len(audit["verdict_mismatches"]),
        len(audit["final_mismatches"]),
        len(audit["summary_mismatches"]),
        f"{audit['response_hash_count']}/{len(audit['iterations'])}",
        len(audit["hash_mismatches"]),
    ])
display(Markdown(table(
    ["Experiment", "Manifest hash", "Dup", "Missing keys", "Missing files",
     "Faults passing", "Verdict mismatch", "Final mismatch", "Summary mismatch",
     "Hashed responses", "Hash mismatch"],
    integrity_rows,
)))
assert all(row[1] is True for row in integrity_rows)
assert all(all(value == 0 for value in row[2:9]) for row in integrity_rows)
assert all(row[10] == 0 for row in integrity_rows)
assert len(all_response_ids) == len(set(all_response_ids))
print("unique stored response IDs:", len(all_response_ids), "/", len(all_response_ids))
print("Note: value_primary predates full response-ID/output-text provenance storage.")

| Experiment | Manifest hash | Dup | Missing keys | Missing files | Faults passing | Verdict mismatch | Final mismatch | Summary mismatch | Hashed responses | Hash mismatch |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| value_primary | True | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0/35 | 0 |
| blind_strict | True | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 22/22 | 0 |
| blind_unit | True | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 21/21 | 0 |
| blind_label | True | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 26/26 | 0 |
| blind_combined | True | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 20/20 | 0 |
| holdout_strict | True | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 27/27 | 0 |
| holdout_combined | True | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 20/20 | 0 |

unique stored response IDs: 136 / 136
Note: value_primary predates full response-ID/output-text provenance storage.


### 6. Blindness checks and the remaining numeric failure

In [6]:
blind_labels = [
    "blind_strict", "blind_unit", "blind_label", "blind_combined",
    "holdout_strict", "holdout_combined",
]
leaked_fault_names = []
for label in blind_labels:
    audit = audits[label]
    for case in audit["manifest"]["cases"]:
        if case.get("operation") != "replace_with_resistor":
            continue
        faulty_path = audit["experiment_root"] / case["fault_id"] / "faulty.cir"
        fault_lines = [
            line for line in faulty_path.read_text(encoding="utf-8").splitlines()
            if line.startswith("RFAULT")
        ]
        for line in fault_lines:
            if not line.startswith("RFAULT1 "):
                leaked_fault_names.append((label, case["fault_id"], line))
print("non-neutral reactive fault names:", leaked_fault_names)
assert not leaked_fault_names

failed_holdout = []
root = audits["holdout_combined"]["experiment_root"]
for final_path in sorted(root.glob("*/trajectory_*/final.json")):
    final = load_json(final_path)
    if not final["final_recovered"]:
        iteration_paths = sorted(final_path.parent.glob("iteration_*.json"))
        failed_holdout.append({
            "fault_id": final["fault_id"],
            "trajectory": final["trajectory"],
            "values": [load_json(path).get("proposed_value") for path in iteration_paths],
            "outcomes": [load_json(path).get("outcome") for path in iteration_paths],
        })
print(json.dumps(failed_holdout, indent=2))
assert failed_holdout == [{
    "fault_id": "holdout_l2_short",
    "trajectory": 2,
    "values": ["3.9788736uH", "3.9788736u", "3.9788736uH"],
    "outcomes": ["SPECIFICATION_VIOLATION"] * 3,
}]

non-neutral reactive fault names: []
[
  {
    "fault_id": "holdout_l2_short",
    "trajectory": 2,
    "values": [
      "3.9788736uH",
      "3.9788736u",
      "3.9788736uH"
    ],
    "outcomes": [
      "SPECIFICATION_VIOLATION",
      "SPECIFICATION_VIOLATION",
      "SPECIFICATION_VIOLATION"
    ]
  }
]


## Takeaways

1. **Ready to share with caveats after paper correction.** All seven frozen runs
   are complete, their manifest hashes match, and every saved accepted simulation
   verdict reconciles with the deterministic validator.
2. **Use 37/48, not 42/48, for primary full diagnosis.** The latter mixed in the
   answer-leaking structural v1.0 run. Recovery and localization totals remain
   46/48 and 45/48.
3. **The interface result reproduces within the tested circuit.** Combined holdout
   reaches 18/18 diagnosis and 17/18 recovery versus 9/18 and 14/18 under strict.
4. **Do not claim cross-circuit or hardware generalization.** All cases are
   simulator-screened synthetic single faults on one ideal 100 MHz lumped filter.
5. **Structured output is not a physical oracle.** The combined interface accepted
   a syntactically valid 1000x value error; only ngspice plus the validator rejected it.
6. **Provenance is strongest for blind runs.** Blind and holdout iterations preserve
   response IDs, output text, and SHA-256 hashes. The older value-primary run retains
   parsed fields and usage but predates full raw-response provenance storage.